# Gold Layer — Risk, Fraud & Portfolio KPI Aggregations
Daily transaction summary, fraud analysis, credit risk distribution, segment portfolio, weekly trends, account scorecards.

In [ ]:
from pyspark.sql.functions import (
    col, sum as _sum, avg as _avg, count, countDistinct,
    min as _min, max as _max, round as _round, when, lit,
    current_timestamp, weekofyear, month
)

In [ ]:
# === TABLE 1: Daily Transaction Summary ===
df_txn = spark.read.format('delta').table('silver_transactions')

gold_daily = (
    df_txn
    .groupBy('transaction_date_only', 'transaction_type', 'merchant_category', 'channel')
    .agg(
        count('*').alias('transaction_count'),
        _sum('amount').alias('total_amount'),
        _avg('amount').alias('avg_amount'),
        _max('amount').alias('max_amount'),
        _sum(col('is_flagged_fraud').cast('int')).alias('fraud_count'),
        _sum(col('is_high_value').cast('int')).alias('high_value_count'),
    )
    .withColumn('fraud_rate', _round(col('fraud_count') / col('transaction_count') * 100, 3))
    .withColumn('total_amount', _round('total_amount', 2))
    .withColumn('avg_amount', _round('avg_amount', 2))
    .withColumn('transaction_week', weekofyear(col('transaction_date_only').cast('date')))
    .withColumn('transaction_month', month(col('transaction_date_only').cast('date')))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_daily.write.mode('overwrite').format('delta').saveAsTable('gold_daily_transaction_summary')
print(f'Gold daily transaction summary: {spark.table("gold_daily_transaction_summary").count()} rows')

In [ ]:
# === TABLE 2: Fraud Analysis by Category & Country ===
gold_fraud = (
    df_txn
    .groupBy('merchant_category', 'country', 'risk_band')
    .agg(
        count('*').alias('total_transactions'),
        _sum(col('is_flagged_fraud').cast('int')).alias('fraud_count'),
        _sum('amount').alias('total_amount'),
        _sum(when(col('is_flagged_fraud'), col('amount')).otherwise(0)).alias('fraud_amount'),
    )
    .withColumn('fraud_rate', _round(col('fraud_count') / col('total_transactions') * 100, 3))
    .withColumn('fraud_amount_pct', _round(col('fraud_amount') / col('total_amount') * 100, 2))
    .withColumn('total_amount', _round('total_amount', 2))
    .withColumn('fraud_amount', _round('fraud_amount', 2))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_fraud.write.mode('overwrite').format('delta').saveAsTable('gold_fraud_analysis')
print(f'Gold fraud analysis: {spark.table("gold_fraud_analysis").count()} rows')

In [ ]:
# === TABLE 3: Credit Risk Distribution ===
df_acct = spark.read.format('delta').table('silver_accounts')
df_cust = spark.read.format('delta').table('silver_customers')

gold_risk = (
    df_acct.join(df_cust, 'customer_id', 'left')
    .groupBy('risk_tier', 'segment', 'account_type', 'utilisation_band')
    .agg(
        count('*').alias('account_count'),
        _sum('balance').alias('total_balance'),
        _avg('balance').alias('avg_balance'),
        _avg('credit_utilisation_pct').alias('avg_utilisation'),
        _sum(when(col('status') == 'Active', 1).otherwise(0)).alias('active_accounts'),
    )
    .withColumn('total_balance', _round('total_balance', 2))
    .withColumn('avg_balance', _round('avg_balance', 2))
    .withColumn('avg_utilisation', _round('avg_utilisation', 2))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_risk.write.mode('overwrite').format('delta').saveAsTable('gold_credit_risk_distribution')
print(f'Gold credit risk distribution: {spark.table("gold_credit_risk_distribution").count()} rows')

In [ ]:
# === TABLE 4: Segment Portfolio ===
gold_segment = (
    df_txn.join(df_cust, 'customer_id', 'left')
    .groupBy('segment', 'region', 'risk_tier')
    .agg(
        countDistinct('customer_id').alias('unique_customers'),
        count('*').alias('total_transactions'),
        _sum('amount').alias('total_volume'),
        _avg('amount').alias('avg_transaction_value'),
        _sum(col('is_flagged_fraud').cast('int')).alias('fraud_count'),
    )
    .withColumn('fraud_rate', _round(col('fraud_count') / col('total_transactions') * 100, 3))
    .withColumn('total_volume', _round('total_volume', 2))
    .withColumn('avg_transaction_value', _round('avg_transaction_value', 2))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_segment.write.mode('overwrite').format('delta').saveAsTable('gold_segment_portfolio')
print(f'Gold segment portfolio: {spark.table("gold_segment_portfolio").count()} rows')

In [ ]:
# === TABLE 5: Weekly Trends ===
weekly = (
    df_txn
    .withColumn('transaction_week', weekofyear(col('transaction_date_only').cast('date')))
    .groupBy('transaction_week', 'transaction_type')
    .agg(
        count('*').alias('weekly_count'),
        _sum('amount').alias('weekly_volume'),
        _sum(col('is_flagged_fraud').cast('int')).alias('weekly_fraud'),
    )
    .withColumn('fraud_rate', _round(col('weekly_fraud') / col('weekly_count') * 100, 3))
    .withColumn('weekly_volume', _round('weekly_volume', 2))
)

gold_weekly = (
    weekly
    .withColumn('gold_timestamp', current_timestamp())
)

gold_weekly.write.mode('overwrite').format('delta').saveAsTable('gold_weekly_trends')
print(f'Gold weekly trends: {spark.table("gold_weekly_trends").count()} rows')

In [ ]:
# === TABLE 6: Account Scorecards ===
gold_scorecard = (
    df_txn.join(df_acct, 'account_id', 'left')
    .groupBy('account_id', 'account_type', 'status')
    .agg(
        count('*').alias('total_transactions'),
        _sum('amount').alias('total_volume'),
        _avg('amount').alias('avg_transaction'),
        _sum(col('is_flagged_fraud').cast('int')).alias('fraud_count'),
        _sum(col('is_high_value').cast('int')).alias('high_value_count'),
    )
    .withColumn('fraud_rate', _round(col('fraud_count') / col('total_transactions') * 100, 3))
    .withColumn('total_volume', _round('total_volume', 2))
    .withColumn('avg_transaction', _round('avg_transaction', 2))
    .withColumn('account_risk',
        when(col('fraud_rate') == 0, 'Clean')
        .when(col('fraud_rate') <= 1, 'Low Risk')
        .when(col('fraud_rate') <= 3, 'Medium Risk')
        .otherwise('High Risk'))
    .withColumn('gold_timestamp', current_timestamp())
)

gold_scorecard.write.mode('overwrite').format('delta').saveAsTable('gold_account_scorecards')
print(f'Gold account scorecards: {spark.table("gold_account_scorecards").count()} rows')